# Build Riffs

This notebook builds the riff-style visualizations that are presented in the final project notebook.

GitHub notebook link: `https://github.com/gknapp101/ds5001_final_project/blob/main/build_riffs.ipynb`

The final project presents the riffs in this order:

1. `Riff 1`: `Word2Vec t-SNE + Sentiment`
2. `Riff 2`: `LDA + Sentiment by Region`
3. `Riff 3`: `PCA + Sentiment`

This build notebook keeps a workflow-oriented order for construction: PCA first, then LDA, then Word2Vec. The interpretation and numbering in `FinalProject.ipynb` should therefore be read as `Word2Vec -> LDA -> PCA`, even though the code below is organized `PCA -> LDA -> Word2Vec`.

The PCA and LDA riffs start from article-level model outputs using `country_id` and `article_n`, but their plotted values are summarized at the country or topic level for readability. The Word2Vec riff works at the term level using `term_str`, so its regime coloring is an overlay inferred from corpus usage rather than something learned directly by the embedding.


In [59]:
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

ROOT = Path.cwd()

PCA_DCM_PATH = ROOT / 'pca_dcm.parquet'
DOC_SENT_PATH = ROOT / 'doc_sent.parquet'
LDA_THETA_PATH = ROOT / 'lda_theta.parquet'
LDA_TOPICS_PATH = ROOT / 'lda_topics.parquet'
VOCAB_W2V_TSNE_PATH = ROOT / 'vocab_w2v_tsne.parquet'
VOCAB_SENT_PATH = ROOT / 'vocab_sent.parquet'
BOW_PATH = ROOT / 'bow.parquet'
LIB_PATH = ROOT / 'lib.csv'

PCA_SENTIMENT_HTML = ROOT / 'riff_pca_sentiment_scatter.html'
LDA_SENTIMENT_HTML = ROOT / 'riff_lda_sentiment_heatmap.html'
LDA_SENTIMENT_HTMLS = [
    ROOT / 'riff_lda_sentiment_heatmap_1.html',
    ROOT / 'riff_lda_sentiment_heatmap_2.html',
    ROOT / 'riff_lda_sentiment_heatmap_3.html',
]
W2V_SENTIMENT_HTML = ROOT / 'riff_w2v_sentiment_tsne.html'
W2V_SENTIMENT_MIXED_HTML = ROOT / 'riff_w2v_sentiment_tsne_mixed.html'

In [60]:
pca_dcm = pd.read_parquet(PCA_DCM_PATH).reset_index()
doc_sent = pd.read_parquet(DOC_SENT_PATH)
lda_theta = pd.read_parquet(LDA_THETA_PATH)
lda_topics = pd.read_parquet(LDA_TOPICS_PATH)
vocab_w2v_tsne = pd.read_parquet(VOCAB_W2V_TSNE_PATH)
vocab_sent = pd.read_parquet(VOCAB_SENT_PATH)
bow = pd.read_parquet(BOW_PATH)
lib = pd.read_csv(LIB_PATH)

topic_cols = [c for c in lda_theta.columns if c.startswith('T')]
sentiment_cols = [
    'negative_mean',
    'positive_mean',
    'uncertainty_mean',
    'litigious_mean',
    'strong_modal_mean',
    'weak_modal_mean',
    'constraining_mean',
    'complexity_mean',
    'sentiment_mean',
]
riff_sentiment_cols = [
    'negative_mean',
    'positive_mean',
    'uncertainty_mean',
    'litigious_mean',
    'constraining_mean',
]
riff_sentiment_label_map = {
    'negative_mean': 'Negative or Adverse Language',
    'positive_mean': 'Positive or Affirming Language',
    'uncertainty_mean': 'Uncertainty and Ambiguity',
    'litigious_mean': 'Legal Dispute and Litigation Language',
    'constraining_mean': 'Restriction and Constraint Language',
}
term_sentiment_cols = [
    'negative',
    'positive',
    'uncertainty',
    'litigious',
    'strong_modal',
    'weak_modal',
    'constraining',
    'complexity',
    'sentiment',
    'matched_lexicon',
]

print('pca_dcm:', pca_dcm.shape)
print('doc_sent:', doc_sent.shape)
print('lda_theta:', lda_theta.shape)
print('vocab_w2v_tsne:', vocab_w2v_tsne.shape)
print('vocab_sent:', vocab_sent.shape)
print('bow:', bow.shape)
print('lib:', lib.shape)

pca_dcm: (33726, 3008)
doc_sent: (33725, 40)
lda_theta: (33220, 22)
vocab_w2v_tsne: (5622, 6)
vocab_sent: (15294, 26)
bow: (1775569, 5)
lib: (192, 19)


## Build Step 1: PCA + Sentiment

This riff joins the PCA document-component matrix to the document-level sentiment table, averages both at the country level, and plots each constitution in PCA space. Instead of showing only one sentiment measure, it creates a separate subplot for each of the five sentiment dimensions used in the Word2Vec riff.

That aggregation is justified because the corpus contains more than 33,000 article-level observations, and a fully article-level scatterplot would be crowded and difficult to compare against constitution-level metadata. The limitation is that constitutions with strongly contrasting articles can look more moderate once those article-level PCA and sentiment values are averaged into a single point.

Because all five panels share the same `PC0` and `PC1` axes, the riff makes it easier to compare how different kinds of constitutional tone are distributed across the same thematic structure.


In [61]:
pca_country = (
    pca_dcm.groupby('country_id')[['PC0', 'PC1']]
    .mean()
    .reset_index()
)

doc_sent_country = (
    doc_sent.groupby('country_id', as_index=False)
    .agg({
        'doc_id': 'first',
        'region_compressed': 'first',
        'v2x_rule': 'first',
        'v2x_rule_cat': 'first',
        **{col: 'mean' for col in sentiment_cols},
    })
)

pca_sent = pca_country.merge(doc_sent_country, on='country_id', how='inner')
pca_riff_cols = riff_sentiment_cols.copy()
pca_riff_labels = [riff_sentiment_label_map[col] for col in pca_riff_cols]

print('Merged country-average PCA + sentiment shape:', pca_sent.shape)
pca_sent[['country_id', 'PC0', 'PC1', *pca_riff_cols]].head(10)


Merged country-average PCA + sentiment shape: (192, 16)


,country_id,PC0,PC1,negative_mean,positive_mean,uncertainty_mean,litigious_mean,constraining_mean
0,Afghanistan,0.026638,-0.017579,0.038714,0.011226,0.001652,0.103653,0.010558
1,Albania,0.012192,-0.005823,0.041348,0.006844,0.017212,0.102552,0.013810
2,Algeria,0.026296,-0.013678,0.034540,0.008430,0.014087,0.071258,0.008915
3,Andorra,0.018075,-0.002018,0.031243,0.006373,0.019726,0.102477,0.009467
4,Angola,0.050340,0.002382,0.033287,0.007926,0.010263,0.099264,0.010678
5,Antigua and Barbuda,-0.083197,0.076171,0.041232,0.005350,0.020394,0.060719,0.006813
6,Argentina,-0.011210,-0.043198,0.025678,0.010524,0.022975,0.073775,0.012648
7,Armenia,0.022234,-0.012425,0.028277,0.006937,0.013334,0.099863,0.014226
8,Australia,-0.051694,0.003400,0.028292,0.004348,0.023012,0.068622,0.010024
9,Austria,-0.005447,-0.003337,0.026464,0.005469,0.007737,0.077169,0.009333


In [62]:
pca_sent_fig = make_subplots(
    rows=3,
    cols=2,
    subplot_titles=pca_riff_labels,
    shared_xaxes=True,
    shared_yaxes=True,
    horizontal_spacing=0.08,
    vertical_spacing=0.08,
)

for i, sentiment_metric in enumerate(pca_riff_cols, start=1):
    row = (i - 1) // 2 + 1
    col = (i - 1) % 2 + 1
    show_x = row == 3
    show_y = col == 1
    colorbar_x = 0.44 if col == 1 else 1.02
    colorbar_y = 1 - ((row - 0.5) / 3)

    pca_sent_fig.add_trace(
        go.Scatter(
            x=pca_sent['PC0'],
            y=pca_sent['PC1'],
            mode='markers',
            marker={
                'size': 9,
                'opacity': 0.75,
                'color': pca_sent[sentiment_metric],
                'colorscale': 'RdBu_r',
                'showscale': True,
                'colorbar': {
                    'title': {'text': 'Avg score'},
                    'len': 0.22,
                    'thickness': 10,
                    'x': colorbar_x,
                    'y': colorbar_y,
                },
            },
            customdata=pca_sent[['country_id', 'doc_id', 'region_compressed', 'v2x_rule_cat', sentiment_metric]].to_numpy(),
            hovertemplate=(
                'Country: %{customdata[0]}<br>'
                'Document: %{customdata[1]}<br>'
                'Region: %{customdata[2]}<br>'
                'Rule-of-law category: %{customdata[3]}<br>'
                + riff_sentiment_label_map[sentiment_metric]
                + ': %{customdata[4]:.4f}<br>'
                'PC0: %{x:.4f}<br>'
                'PC1: %{y:.4f}<extra></extra>'
            ),
            showlegend=False,
        ),
        row=row,
        col=col,
    )

    pca_sent_fig.update_xaxes(
        title_text='PC0' if show_x else None,
        showticklabels=show_x,
        row=row,
        col=col,
    )
    pca_sent_fig.update_yaxes(
        title_text='PC1' if show_y else None,
        showticklabels=show_y,
        row=row,
        col=col,
    )

pca_sent_fig.update_layout(
    template='plotly_white',
    title='PCA + Sentiment: country-average PC0 vs PC1 by sentiment dimension',
    height=1200,
    width=1400,
    margin={'l': 40, 'r': 90, 't': 90, 'b': 40},
)

pca_sent_fig.show()


In [63]:
pca_sent_fig.write_html(PCA_SENTIMENT_HTML, include_plotlyjs='cdn')
print(PCA_SENTIMENT_HTML)


c:\Users\garre\school\spring_2026\ds_5001\riff_pca_sentiment_scatter.html


## Build Step 2: LDA + Sentiment by Region

This riff joins the LDA theta table to the document-level sentiment table. It assigns each article a dominant topic based on the largest topic weight, then computes mean sentiment values by dominant topic within each `region_compressed` group.

Because the full set of regional panels is dense, the notebook breaks the output into three separate heatmap figures. Each figure contains a smaller set of regional subplots so the topic labels and sentiment patterns are easier to read.

The heatmaps use the same five sentiment dimensions as the Word2Vec riff: negative, positive, uncertainty, litigious, and constraining. Within each regional panel, warmer cells mean that articles dominated by that topic tend to carry more of that sentiment dimension on average, while lighter cells mean that dimension appears less strongly for that topic.


In [64]:
lda_sent = lda_theta.merge(
    doc_sent[
        [
            'country_id', 'article_n', 'doc_id', 'region_compressed', 'v2x_rule_cat',
            *riff_sentiment_cols,
        ]
    ],
    on=['country_id', 'article_n'],
    how='inner',
)

lda_sent['dominant_topic'] = lda_sent[topic_cols].idxmax(axis=1)
lda_sent['dominant_topic_weight'] = lda_sent[topic_cols].max(axis=1)

topic_lookup = lda_topics.reset_index()[['topic_id', 'top_terms']].copy()
topic_lookup['topic_n'] = topic_lookup['topic_id'].str.extract(r'(\d+)').astype(int)
topic_lookup = topic_lookup.sort_values('topic_n').copy()
topic_lookup['topic_label'] = topic_lookup['topic_id'] + ': ' + topic_lookup['top_terms']
topic_label_map = dict(zip(topic_lookup['topic_id'], topic_lookup['topic_label']))
lda_sent['topic_label'] = lda_sent['dominant_topic'].map(topic_label_map).fillna(lda_sent['dominant_topic'])

topic_sentiment_region = (
    lda_sent.groupby(['region_compressed', 'dominant_topic'])[riff_sentiment_cols]
    .mean()
    .reset_index()
)
topic_sentiment_region['topic_label'] = topic_sentiment_region['dominant_topic'].map(topic_label_map)

region_order = sorted(topic_sentiment_region['region_compressed'].dropna().unique())
region_groups = [
    region_order[:4],
    region_order[4:7],
    region_order[7:],
]
topic_order = topic_lookup['topic_id'].tolist()
topic_label_order = topic_lookup['topic_label'].tolist()

region_mats = {}
for region in region_order:
    region_mat = (
        topic_sentiment_region.loc[topic_sentiment_region['region_compressed'] == region]
        .set_index('dominant_topic')[riff_sentiment_cols]
        .reindex(topic_order)
    )
    region_mat = region_mat.rename(columns=riff_sentiment_label_map)
    region_mat.index = topic_label_order
    region_mat.index.name = 'topic_label'
    region_mats[region] = region_mat

topic_sentiment = pd.concat(region_mats, names=['region_compressed', 'topic_label'])

print('Merged LDA + sentiment shape:', lda_sent.shape)
topic_sentiment.head(len(topic_label_order)).round(4)


Merged LDA + sentiment shape: (33220, 33)


Negative or Adverse Language  \
region_compressed topic_label                                                                        
Asia              T00: days period law date time day force                                  0.0276   
                  T01: year expenditure motion law purposes money...                        0.0105   
                  T02: law laws courts matters jurisdiction decis...                        0.0239   
                  T03: information freedom law persons right expr...                        0.0320   
                  T04: office person functions accordance advice ...                        0.0227   
                  T05: territory treaties committees agreements l...                        0.0080   
                  T06: duties functions law exercise members acco...                        0.0267   
                  T07: force members law para constitution meetin...                        0.0150   
                  T08: state emergency war declaration forces mea...                        0.0186   
                  T09: court person proceedings offence law case ...                        0.0660   
                  T10: members majority vote votes election candi...                        0.0174   
                  T11: person law authority order religion extent...                        0.0435   
                  T12: citizen person property law right citizens...                        0.0365   
                  T13: office member election members term years ...                        0.0297   
                  T14: law right cases person act rights case                               0.1101   
                  T15: rights people freedoms law citizens princi...                        0.0231   
                  T16: right parties work conditions party law ci...                        0.0219   
                  T17: government service power law authority pow...                        0.0114   
                  T18: development education law resources servic...                        0.0170   
                  T19: members number session sessions referendum...                        0.0191   

                                                                      Positive or Affirming Language  \
region_compressed topic_label                                                                          
Asia              T00: days period law date time day force                                    0.0025   
                  T01: year expenditure motion law purposes money...                          0.0030   
                  T02: law laws courts matters jurisdiction decis...                          0.0066   
                  T03: information freedom law persons right expr...                          0.0127   
                  T04: office person functions accordance advice ...                          0.0050   
                  T05: territory treaties committees agreements l...                          0.0055   
                  T06: duties functions law exercise members acco...                          0.0112   
                  T07: force members law para constitution meetin...                          0.0070   
                  T08: state emergency war declaration forces mea...                          0.0044   
                  T09: court person proceedings offence law case ...                          0.0036   
                  T10: members majority vote votes election candi...                          0.0024   
                  T11: person law authority order religion extent...                          0.0067   
                  T12: citizen person property law right citizens...                          0.0083   
                  T13: office member election members term years ...                          0.0045   
                  T14: law right cases person act rights case                                 0.0054   
                  T15: rights people freedoms law citizen

In [65]:
ncols = 2
zmin = min(mat.min().min() for mat in region_mats.values())
zmax = max(mat.max().max() for mat in region_mats.values())

lda_sent_figs = []
for fig_idx, regions in enumerate(region_groups, start=1):
    n_regions = len(regions)
    nrows = (n_regions + ncols - 1) // ncols
    fig = make_subplots(
        rows=nrows,
        cols=ncols,
        subplot_titles=regions,
        horizontal_spacing=0.10,
        vertical_spacing=0.08,
        shared_xaxes=True,
        shared_yaxes=True,
    )

    for i, region in enumerate(regions, start=1):
        row = (i - 1) // ncols + 1
        col = (i - 1) % ncols + 1
        region_mat = region_mats[region]
        fig.add_trace(
            go.Heatmap(
                z=region_mat.to_numpy(),
                x=region_mat.columns.tolist(),
                y=region_mat.index.tolist(),
                colorscale='YlOrRd',
                zmin=zmin,
                zmax=zmax,
                coloraxis='coloraxis',
                hovertemplate='Region: ' + region + '<br>Sentiment dimension: %{x}<br>Dominant topic: %{y}<br>Average score: %{z:.4f}<extra></extra>',
            ),
            row=row,
            col=col,
        )
        show_y = col == 1
        show_x = row == nrows
        fig.update_yaxes(
            autorange='reversed',
            showticklabels=show_y,
            title_text='Dominant topic' if show_y else None,
            row=row,
            col=col,
        )
        fig.update_xaxes(
            showticklabels=show_x,
            title_text='Sentiment dimension' if show_x else None,
            tickangle=35,
            row=row,
            col=col,
        )

    fig.update_layout(
        template='plotly_white',
        title=f'LDA + Sentiment by Region: average sentiment by dominant topic (Figure {fig_idx})',
        coloraxis={'colorscale': 'YlOrRd', 'cmin': zmin, 'cmax': zmax, 'colorbar': {'title': 'Average score'}},
        height=460 * nrows,
        width=1600,
        margin={'l': 40, 'r': 40, 't': 90, 'b': 40},
    )
    lda_sent_figs.append(fig)
    fig.show()


In [66]:
for path, fig in zip(LDA_SENTIMENT_HTMLS, lda_sent_figs):
    fig.write_html(path, include_plotlyjs='cdn')
    print(path)

c:\Users\garre\school\spring_2026\ds_5001\riff_lda_sentiment_heatmap_1.html
c:\Users\garre\school\spring_2026\ds_5001\riff_lda_sentiment_heatmap_2.html
c:\Users\garre\school\spring_2026\ds_5001\riff_lda_sentiment_heatmap_3.html


## Build Step 3: Word2Vec t-SNE + Sentiment

This riff keeps the sentiment-based facet layout, but colors each term by the regime category with which it is most strongly associated in the corpus. Each term is matched to the constitutions where it appears through the long BOW table, then assigned a dominant `v2x_regime_cat`.

Dominant regime is calculated using the mean term count per country within each regime category rather than raw pooled counts alone. That normalization is justified because it reduces the bias that would otherwise let larger regime groups dominate simply by having more constitutions. Those regime scores are then normalized within each term so the hover can show how strongly a word leans toward its assigned regime relative to the alternatives.

This coloring should still be read as an external overlay rather than as something encoded directly in the embedding. The t-SNE positions reflect semantic similarity among terms, while the regime assignment summarizes which constitutions use a term most strongly on average.

This version focuses on the clearest legal-tone dimensions: negative, positive, uncertainty, litigious, and constraining.

To make the cross-regime overlap easier to see, the notebook also builds a second version of the chart that relabels weakly dominated terms as `Mixed / Shared usage`. In that companion chart, the label includes the top two regimes sharing the term most strongly, and a term is treated as mixed when its strongest regime score is still relatively low or when the top two regime scores are very close to each other.


In [67]:
regime_lookup = (
    lib[['country_id', 'v2x_regime_cat']]
    .drop_duplicates(subset=['country_id'])
    .dropna(subset=['v2x_regime_cat'])
    .copy()
)

bow_regime = bow[['country_id', 'term_str', 'n']].merge(
    regime_lookup,
    on='country_id',
    how='inner',
)

term_regime_country = (
    bow_regime.groupby(['term_str', 'v2x_regime_cat', 'country_id'], as_index=False)['n']
    .sum()
)

term_regime_scores = (
    term_regime_country.groupby(['term_str', 'v2x_regime_cat'], as_index=False)['n']
    .mean()
    .rename(columns={'n': 'mean_term_count'})
)

term_regime_scores['normalized_regime_score'] = (
    term_regime_scores['mean_term_count']
    / term_regime_scores.groupby('term_str')['mean_term_count'].transform('sum')
)

term_regime_ranked = term_regime_scores.sort_values(
    ['term_str', 'normalized_regime_score', 'mean_term_count'],
    ascending=[True, False, False],
).copy()

dominant_term_regime = term_regime_ranked.groupby('term_str', as_index=False).first()
dominant_term_regime['dominant_regime'] = dominant_term_regime['v2x_regime_cat']

# Mark terms as mixed when no regime clearly dominates their normalized usage profile.
top_two_regimes = term_regime_ranked.groupby('term_str').head(2).copy()
regimes_present_summary = (
    term_regime_ranked.groupby('term_str')['v2x_regime_cat']
    .apply(lambda s: ' + '.join(dict.fromkeys(s.tolist())))
    .reset_index(name='regimes_present')
)
top_two_label_summary = (
    top_two_regimes.groupby('term_str')['v2x_regime_cat']
    .apply(lambda s: ' + '.join(s.tolist()))
    .reset_index(name='top_two_regimes_label')
)
regime_mix_summary = (
    top_two_regimes.groupby('term_str')
    .agg(
        top_score=('normalized_regime_score', 'max'),
        second_score=(
            'normalized_regime_score',
            lambda s: s.nlargest(2).iloc[-1] if len(s) > 1 else 0,
        ),
    )
    .reset_index()
)
regime_mix_summary['score_gap'] = (
    regime_mix_summary['top_score'] - regime_mix_summary['second_score']
)
regime_mix_summary = regime_mix_summary.merge(
    dominant_term_regime[['term_str', 'dominant_regime']],
    on='term_str',
    how='left',
)
regime_mix_summary = regime_mix_summary.merge(
    regimes_present_summary,
    on='term_str',
    how='left',
)
regime_mix_summary = regime_mix_summary.merge(
    top_two_label_summary,
    on='term_str',
    how='left',
)
regime_mix_summary['regime_mix_label'] = regime_mix_summary['dominant_regime']
regime_mix_summary.loc[
    (regime_mix_summary['top_score'] < 0.40)
    | (regime_mix_summary['score_gap'] < 0.10),
    'regime_mix_label',
] = (
    'Mixed / Shared usage: '
    + regime_mix_summary.loc[
        (regime_mix_summary['top_score'] < 0.40)
        | (regime_mix_summary['score_gap'] < 0.10),
        'top_two_regimes_label',
    ]
)

w2v_sent = vocab_w2v_tsne.merge(
    vocab_sent[['term_str', *term_sentiment_cols]],
    on='term_str',
    how='left',
).merge(
    dominant_term_regime[
        ['term_str', 'dominant_regime', 'mean_term_count', 'normalized_regime_score']
    ],
    on='term_str',
    how='left',
).merge(
    regime_mix_summary[['term_str', 'top_score', 'second_score', 'score_gap', 'regime_mix_label', 'regimes_present', 'top_two_regimes_label']],
    on='term_str',
    how='left',
)

w2v_sent = w2v_sent[w2v_sent['matched_lexicon'] == 1].copy()
w2v_sent = w2v_sent.dropna(subset=['dominant_regime']).copy()
w2v_sent = w2v_sent[w2v_sent['dominant_regime'] != 'Unknown'].copy()
term_category_cols = [
    'negative', 'positive', 'uncertainty', 'litigious', 'constraining',
]

sentiment_labels = {
    'negative': 'Negative or Adverse Language',
    'positive': 'Positive or Affirming Language',
    'uncertainty': 'Uncertainty and Ambiguity',
    'litigious': 'Legal Dispute and Litigation Language',
    'constraining': 'Restriction and Constraint Language',
}

w2v_sent_long = w2v_sent.melt(
    id_vars=[
        'term_str',
        'n',
        'df',
        'x',
        'y',
        'sentiment',
        'dominant_regime',
        'mean_term_count',
        'normalized_regime_score',
        'top_score',
        'second_score',
        'score_gap',
        'regime_mix_label',
        'regimes_present',
        'top_two_regimes_label',
    ],
    value_vars=term_category_cols,
    var_name='sentiment_dimension',
    value_name='is_present',
)
w2v_sent_long = w2v_sent_long[w2v_sent_long['is_present'] == 1].copy()
w2v_sent_long['sentiment_label'] = w2v_sent_long['sentiment_dimension'].map(sentiment_labels)

print('Merged Word2Vec t-SNE + sentiment shape:', w2v_sent.shape)
print('Long sentiment facet table shape:', w2v_sent_long.shape)
w2v_sent_long[['term_str', 'x', 'y', 'n', 'sentiment_dimension', 'sentiment_label', 'dominant_regime', 'regime_mix_label', 'regimes_present', 'normalized_regime_score']].head(10)

Merged Word2Vec t-SNE + sentiment shape: (4455, 25)
Long sentiment facet table shape: (814, 18)


,term_str,x,y,n,sentiment_dimension,sentiment_label,dominant_regime,regime_mix_label,regimes_present,normalized_regime_score
0,abandoned,-17.639801,-20.022953,35,negative,Negative or Adverse Language,Liberal democracy,Mixed / Shared usage: Liberal democracy + Elec...,Liberal democracy + Electoral autocracy + Elec...,0.266667
1,abandonment,-22.354099,-11.046404,40,negative,Negative or Adverse Language,Closed autocracy,Mixed / Shared usage: Closed autocracy + Elect...,Closed autocracy + Electoral democracy + Elect...,0.304054
7,abolish,37.040607,-18.245388,72,negative,Negative or Adverse Language,Closed autocracy,Mixed / Shared usage: Closed autocracy + Elect...,Closed autocracy + Electoral democracy + Elect...,0.391365
8,abolished,30.679983,-9.317293,104,negative,Negative or Adverse Language,Electoral democracy,Mixed / Shared usage: Electoral democracy + Li...,Electoral democracy + Liberal democracy + Unkn...,0.218369
9,abolishing,-13.782855,-7.971602,20,negative,Negative or Adverse Language,Closed autocracy,Mixed / Shared usage: Closed autocracy + Elect...,Closed autocracy + Electoral autocracy + Elect...,0.200000
15,abrogate,37.735207,-18.996151,33,negative,Negative or Adverse Language,Closed autocracy,Mixed / Shared usage: Closed autocracy + Liber...,Closed autocracy + Liberal democracy + Elector...,0.232153
16,abrogated,48.985531,-26.670279,109,negative,Negative or Adverse Language,Electoral autocracy,Electoral autocracy,Electoral autocracy + Electoral democracy + Li...,0.480916
17,abrogation,40.813507,-27.993004,43,negative,Negative or Adverse Language,Liberal democracy,Mixed / Shared usage: Liberal democracy + Elec...,Liberal democracy + Electoral democracy + Elec...,0.247971
21,abuse,-2.815053,-34.873775,200,negative,Negative or Adverse Language,Electoral democracy,Mixed / Shared usage: Electoral democracy + El...,Electoral democracy + Electoral autocracy + Cl...,0.281435
22,abuses,7.705590,-25.923147,33,negative,Negative or Adverse Language,Closed autocracy,Mixed / Shared usage: Closed autocracy + Elect...,Closed autocracy + Electoral democracy + Liber...,0.328125


In [68]:
regime_order = sorted(w2v_sent_long['dominant_regime'].dropna().unique())
sentiment_label_order = [
    'Negative or Adverse Language',
    'Positive or Affirming Language',
    'Uncertainty and Ambiguity',
    'Legal Dispute and Litigation Language',
    'Restriction and Constraint Language',
]

w2v_sent_fig = px.scatter(
    w2v_sent_long,
    x='x',
    y='y',
    color='dominant_regime',
    category_orders={
        'dominant_regime': regime_order,
        'sentiment_label': sentiment_label_order,
    },
    size='n',
    hover_data={
        'term_str': True,
        'n': True,
        'df': True,
        'sentiment_label': True,
        'sentiment': True,
        'dominant_regime': True,
        'mean_term_count': ':.3f',
        'normalized_regime_score': ':.3f',
        'x': False,
        'y': False,
    },
    facet_col='sentiment_label',
    facet_col_wrap=3,
    title='Word2Vec t-SNE + Sentiment: regime-colored constitutional vocabulary across legal-tone dimensions',
)

w2v_sent_fig.update_traces(marker={'opacity': 0.75})
w2v_sent_fig.update_layout(template='plotly_white')
w2v_sent_fig.for_each_annotation(lambda a: a.update(text=a.text.split('=')[-1]))
w2v_sent_fig.show()

mix_order = ['Mixed / Shared usage'] + [
    label
    for label in sorted(w2v_sent_long['regime_mix_label'].dropna().unique())
    if label.startswith('Mixed / Shared usage')
]
mix_order += [r for r in regime_order if r != 'Mixed / Shared usage']

w2v_sent_mixed_fig = px.scatter(
    w2v_sent_long,
    x='x',
    y='y',
    color='regime_mix_label',
    category_orders={
        'regime_mix_label': mix_order,
        'sentiment_label': sentiment_label_order,
    },
    size='n',
    hover_data={
        'term_str': True,
        'n': True,
        'df': True,
        'sentiment_label': True,
        'sentiment': True,
        'dominant_regime': True,
        'regime_mix_label': True,
        'mean_term_count': ':.3f',
        'normalized_regime_score': ':.3f',
        'top_score': ':.3f',
        'second_score': ':.3f',
        'score_gap': ':.3f',
        'x': False,
        'y': False,
    },
    facet_col='sentiment_label',
    facet_col_wrap=3,
    title='Word2Vec t-SNE + Sentiment: mixed/shared-usage overlay for cross-regime overlap',
)

w2v_sent_mixed_fig.update_traces(marker={'opacity': 0.75})
w2v_sent_mixed_fig.update_layout(template='plotly_white')
w2v_sent_mixed_fig.for_each_annotation(lambda a: a.update(text=a.text.split('=')[-1]))
w2v_sent_mixed_fig.show()

In [69]:
top_words_by_sentiment = (
    w2v_sent_long[
        ['sentiment_label', 'term_str', 'n', 'dominant_regime', 'normalized_regime_score']
    ]
    .drop_duplicates()
    .sort_values(['sentiment_label', 'n', 'normalized_regime_score'], ascending=[True, False, False])
    .groupby('sentiment_label', group_keys=False)
    .head(10)
    .copy()
)

top_words_by_sentiment['rank'] = top_words_by_sentiment.groupby('sentiment_label').cumcount() + 1
top_words_by_sentiment['normalized_regime_score'] = top_words_by_sentiment['normalized_regime_score'].map(lambda x: f'{x:.3f}')

sentiment_table_figs = {}
for sentiment_label in sentiment_label_order:
    table_df = top_words_by_sentiment[top_words_by_sentiment['sentiment_label'] == sentiment_label].copy()
    table_df = table_df.rename(columns={
        'rank': 'Rank',
        'term_str': 'Word',
        'n': 'Frequency',
        'dominant_regime': 'Dominant Regime',
        'normalized_regime_score': 'Normalized Regime Score',
    })[
        ['Rank', 'Word', 'Frequency', 'Dominant Regime', 'Normalized Regime Score']
    ]

    table_fig = go.Figure(
        data=[
            go.Table(
                header=dict(
                    values=list(table_df.columns),
                    align='left',
                    fill_color='#D9E6F2',
                    font=dict(size=11),
                ),
                cells=dict(
                    values=[table_df[col] for col in table_df.columns],
                    align='left',
                    fill_color='white',
                    font=dict(size=10),
                    height=24,
                ),
            )
        ]
    )

    table_fig.update_layout(
        title=f'Top 10 Words: {sentiment_label}',
        height=max(360, 120 + 24 * len(table_df)),
        margin=dict(t=60, r=20, b=20, l=20),
        template='plotly_white',
    )
    table_fig.show()
    sentiment_table_figs[sentiment_label] = table_fig

saved_w2v_tables = {}
w2v_sent_fig.write_html(W2V_SENTIMENT_HTML, include_plotlyjs='cdn')
w2v_sent_mixed_fig.write_html(W2V_SENTIMENT_MIXED_HTML, include_plotlyjs='cdn')
for sentiment_label, table_fig in sentiment_table_figs.items():
    safe_name = sentiment_label.lower().replace(' ', '_').replace(':', '').replace('-', '_')
    table_path = ROOT / f'riff_w2v_sentiment_table_{safe_name}.html'
    table_fig.write_html(table_path, include_plotlyjs='cdn')
    saved_w2v_tables[sentiment_label] = table_path

print('Saved Word2Vec t-SNE chart:', W2V_SENTIMENT_HTML)
print('Saved Word2Vec mixed/shared-usage chart:', W2V_SENTIMENT_MIXED_HTML)
for sentiment_label, table_path in saved_w2v_tables.items():
    print(f'Saved Word2Vec table [{sentiment_label}]: {table_path}')

Saved Word2Vec t-SNE chart: c:\Users\garre\school\spring_2026\ds_5001\riff_w2v_sentiment_tsne.html
Saved Word2Vec mixed/shared-usage chart: c:\Users\garre\school\spring_2026\ds_5001\riff_w2v_sentiment_tsne_mixed.html
Saved Word2Vec table [Negative or Adverse Language]: c:\Users\garre\school\spring_2026\ds_5001\riff_w2v_sentiment_table_negative_or_adverse_language.html
Saved Word2Vec table [Positive or Affirming Language]: c:\Users\garre\school\spring_2026\ds_5001\riff_w2v_sentiment_table_positive_or_affirming_language.html
Saved Word2Vec table [Uncertainty and Ambiguity]: c:\Users\garre\school\spring_2026\ds_5001\riff_w2v_sentiment_table_uncertainty_and_ambiguity.html
Saved Word2Vec table [Legal Dispute and Litigation Language]: c:\Users\garre\school\spring_2026\ds_5001\riff_w2v_sentiment_table_legal_dispute_and_litigation_language.html
Saved Word2Vec table [Restriction and Constraint Language]: c:\Users\garre\school\spring_2026\ds_5001\riff_w2v_sentiment_table_restriction_and_constrai

## Final Project Mapping

- `FinalProject.ipynb` `Riff 1` corresponds to `Word2Vec t-SNE + Sentiment` in this notebook.
- `FinalProject.ipynb` `Riff 2` corresponds to `LDA + Sentiment by Region` in this notebook.
- `FinalProject.ipynb` `Riff 3` corresponds to `PCA + Sentiment` in this notebook.
- The Word2Vec tables beneath the t-SNE break the summary out by sentiment dimension so each panel has its own top-10 word list with regime association.
- All three riffs are designed to connect earlier model outputs back to corpus interpretation rather than to introduce new standalone models.
